## Example 1: Basic file watching setup and configuration

In [ ]:
import watchfiles
import asyncio
import tempfile
import os
from pathlib import Path

# Example 1: Basic file watching setup and configuration
print("Basic file watching with watchfiles:")

# Show available components
print(f"Watchfiles version: {watchfiles.__version__}")
print("Available classes:", [attr for attr in dir(watchfiles) if attr[0].isupper()])

# Create temporary directory for demonstration
temp_dir = tempfile.mkdtemp()
print(f"Created demo directory: {temp_dir}")

# Show different change types
change_types = list(watchfiles.Change)
print(f"Change types monitored: {[c.name for c in change_types]}")

# Demonstrate filter configuration
default_filter = watchfiles.DefaultFilter()
python_filter = watchfiles.PythonFilter()

print(f"\nFilter examples:")
print(f"DefaultFilter: {type(default_filter).__name__}")
print(f"PythonFilter: {type(python_filter).__name__}")

# Test file patterns that would be watched
test_files = [
    "script.py",
    "data.json", 
    "config.yaml",
    "temp.tmp",
    "__pycache__/module.pyc",
    ".git/config"
]

print(f"\nFilter behavior simulation:")
for filename in test_files:
    # Simulate filter checking (DefaultFilter ignores common temp files)
    should_watch = not any(pattern in filename.lower() for pattern in 
                          ['__pycache__', '.git', '.tmp', '.pyc'])
    status = "✓ WATCH" if should_watch else "✗ IGNORE"
    print(f"  {filename}: {status}")

# Show watch function signature and options
print(f"\nWatching functions available:")
print("- watch(): Synchronous file watching")
print("- awatch(): Asynchronous file watching") 
print("- run_process(): Watch and run command on changes")
print("- arun_process(): Async watch and run command")

# Cleanup
os.rmdir(temp_dir)
print(f"\nWatchfiles provides high-performance file system monitoring")

## Example 2: Simulated file watching with change detection

In [ ]:
import watchfiles
import asyncio
import tempfile
import os
import time
from pathlib import Path

# Example 2: Simulated file watching with change detection
print("File watching simulation with change detection:")

class FileWatchSimulator:
    """Simulate file watching for demonstration"""
    
    def __init__(self, watch_path):
        self.watch_path = Path(watch_path)
        self.last_scan = {}
        self.changes = []
    
    def scan_directory(self):
        """Scan directory and detect changes"""
        current_scan = {}
        
        if self.watch_path.exists():
            for file_path in self.watch_path.rglob("*"):
                if file_path.is_file():
                    stat = file_path.stat()
                    current_scan[str(file_path)] = {
                        'mtime': stat.st_mtime,
                        'size': stat.st_size
                    }
        
        # Detect changes
        changes = []
        
        # Check for new or modified files
        for path, info in current_scan.items():
            if path not in self.last_scan:
                changes.append((watchfiles.Change.added, path))
            elif self.last_scan[path]['mtime'] != info['mtime']:
                changes.append((watchfiles.Change.modified, path))
        
        # Check for deleted files
        for path in self.last_scan:
            if path not in current_scan:
                changes.append((watchfiles.Change.deleted, path))
        
        self.last_scan = current_scan
        return changes

# Create demo environment
demo_dir = tempfile.mkdtemp()
simulator = FileWatchSimulator(demo_dir)

print(f"Demo directory: {demo_dir}")

# Initial scan
print(f"\nInitial scan:")
changes = simulator.scan_directory()
print(f"Changes detected: {len(changes)}")

# Simulate file operations
print(f"\nSimulating file operations:")

# Create a file
test_file = Path(demo_dir) / "test.txt"
test_file.write_text("Hello, World!")
print(f"Created: {test_file.name}")

# Scan for changes
changes = simulator.scan_directory()
for change_type, path in changes:
    filename = Path(path).name
    print(f"  {change_type.name.upper()}: {filename}")

# Modify the file
time.sleep(0.01)  # Ensure different mtime
test_file.write_text("Hello, Modified World!")
print(f"\nModified: {test_file.name}")

changes = simulator.scan_directory()
for change_type, path in changes:
    filename = Path(path).name
    print(f"  {change_type.name.upper()}: {filename}")

# Create subdirectory with files
subdir = Path(demo_dir) / "subdir"
subdir.mkdir()
(subdir / "nested.py").write_text("print('nested')")
print(f"\nCreated subdirectory with nested file")

changes = simulator.scan_directory()
for change_type, path in changes:
    filename = Path(path).name
    print(f"  {change_type.name.upper()}: {filename}")

# Delete a file
test_file.unlink()
print(f"\nDeleted: {test_file.name}")

changes = simulator.scan_directory()
for change_type, path in changes:
    filename = Path(path).name
    print(f"  {change_type.name.upper()}: {filename}")

# Cleanup
import shutil
shutil.rmtree(demo_dir)

print(f"\nWatchfiles efficiently detects file system changes")
print("Supports recursive directory monitoring with filtering")

## Example 3: Async file watching patterns

In [ ]:
import watchfiles
import asyncio
import tempfile
import os
import time
from pathlib import Path

# Example 3: Async file watching patterns
print("Async file watching patterns:")

class AsyncFileWatcher:
    """Async file watcher implementation demo"""
    
    def __init__(self, watch_paths, filters=None):
        self.watch_paths = watch_paths if isinstance(watch_paths, list) else [watch_paths]
        self.filters = filters or watchfiles.DefaultFilter()
        self.handlers = {}
        self.running = False
    
    def on(self, change_type, handler):
        """Register event handler for specific change type"""
        if change_type not in self.handlers:
            self.handlers[change_type] = []
        self.handlers[change_type].append(handler)
    
    async def simulate_watch(self, duration=1.0):
        """Simulate async watching for demo purposes"""
        self.running = True
        print(f"Started watching {len(self.watch_paths)} paths")
        
        # Simulate events over time
        events = [
            (0.1, watchfiles.Change.added, "new_file.py"),
            (0.3, watchfiles.Change.modified, "existing.txt"),
            (0.5, watchfiles.Change.added, "another.json"),
            (0.7, watchfiles.Change.deleted, "old_file.log"),
            (0.9, watchfiles.Change.modified, "config.yaml")
        ]
        
        start_time = asyncio.get_event_loop().time()
        
        for delay, change_type, filename in events:
            await asyncio.sleep(delay)
            if not self.running:
                break
                
            # Simulate handling the event
            await self._handle_event(change_type, filename)
            
        print(f"Watch simulation completed")
    
    async def _handle_event(self, change_type, filename):
        """Handle file system event"""
        print(f"Event: {change_type.name} - {filename}")
        
        # Call registered handlers
        if change_type in self.handlers:
            for handler in self.handlers[change_type]:
                try:
                    if asyncio.iscoroutinefunction(handler):
                        await handler(filename)
                    else:
                        handler(filename)
                except Exception as e:
                    print(f"Handler error: {e}")
    
    def stop(self):
        """Stop watching"""
        self.running = False

# Create demo watcher
demo_dir = tempfile.mkdtemp()
watcher = AsyncFileWatcher([demo_dir])

# Register event handlers
async def on_file_added(filename):
    print(f"  → Processing new file: {filename}")
    await asyncio.sleep(0.01)  # Simulate processing time

def on_file_modified(filename):
    print(f"  → Reloading modified file: {filename}")

async def on_file_deleted(filename):
    print(f"  → Cleaning up after deleted file: {filename}")

# Register handlers
watcher.on(watchfiles.Change.added, on_file_added)
watcher.on(watchfiles.Change.modified, on_file_modified)
watcher.on(watchfiles.Change.deleted, on_file_deleted)

print(f"Registered handlers for: {list(watcher.handlers.keys())}")

# Run async watch simulation
await watcher.simulate_watch(duration=1.0)

# Demonstrate real watchfiles async usage pattern
print(f"\nReal watchfiles async usage pattern:")

async def demo_real_awatch():
    """Demonstrate how real awatch would be used"""
    print("Setting up awatch (simulation):")
    
    # This is how you'd use real awatch
    async_watch_code = '''
    async for changes in watchfiles.awatch(demo_dir):
        for change_type, path in changes:
            print(f"{change_type.name}: {path}")
            
            # Process based on change type
            if change_type == watchfiles.Change.added:
                await process_new_file(path)
            elif change_type == watchfiles.Change.modified:
                await reload_file(path)
            elif change_type == watchfiles.Change.deleted:
                await cleanup_file(path)
    '''
    
    print("Code pattern:")
    print(async_watch_code)

await demo_real_awatch()

# Cleanup
import shutil
shutil.rmtree(demo_dir)

print(f"\nAsync watching enables efficient event-driven file processing")
print("Perfect for hot-reloading development servers and build tools")

## Example 4: Process automation with file watching

In [ ]:
import watchfiles
import asyncio
import tempfile
import subprocess
import sys
from pathlib import Path

# Example 4: Process automation with file watching
print("Process automation with file watching:")

class ProcessRunner:
    """File watcher that runs processes on changes"""
    
    def __init__(self, watch_path):
        self.watch_path = Path(watch_path)
        self.commands = {}
        self.running_processes = {}
    
    def on_pattern(self, pattern, command):
        """Register command to run when files matching pattern change"""
        self.commands[pattern] = command
    
    def simulate_process_run(self, command, trigger_file):
        """Simulate running a process"""
        print(f"  Running: {command}")
        print(f"  Triggered by: {trigger_file}")
        
        # Simulate process execution
        if "test" in command:
            result = "Tests passed ✓"
        elif "build" in command:
            result = "Build completed ✓"
        elif "lint" in command:
            result = "Linting completed ✓"
        else:
            result = "Command executed ✓"
        
        print(f"  Result: {result}")
        return result
    
    def should_trigger(self, file_path, pattern):
        """Check if file matches pattern"""
        file_path = Path(file_path)
        
        if pattern == "*.py":
            return file_path.suffix == ".py"
        elif pattern == "*.js":
            return file_path.suffix == ".js"
        elif pattern == "*.yaml" or pattern == "*.yml":
            return file_path.suffix in [".yaml", ".yml"]
        elif pattern == "test_*":
            return file_path.name.startswith("test_")
        else:
            return True
    
    def handle_change(self, change_type, file_path):
        """Handle file system change"""
        print(f"\nFile change detected: {change_type.name} - {Path(file_path).name}")
        
        # Check which commands should run
        for pattern, command in self.commands.items():
            if self.should_trigger(file_path, pattern):
                self.simulate_process_run(command, Path(file_path).name)

# Create demo environment
demo_dir = tempfile.mkdtemp()
runner = ProcessRunner(demo_dir)

# Set up automation rules
automation_rules = {
    "*.py": "python -m pytest tests/",
    "*.js": "npm test",
    "*.yaml": "docker-compose up -d",
    "test_*": "python -m pytest {file}",
}

print(f"Setting up automation rules:")
for pattern, command in automation_rules.items():
    runner.on_pattern(pattern, command)
    print(f"  {pattern} → {command}")

# Simulate file changes and automation
test_scenarios = [
    (watchfiles.Change.modified, "src/main.py"),
    (watchfiles.Change.added, "tests/test_feature.py"),
    (watchfiles.Change.modified, "frontend/app.js"),
    (watchfiles.Change.modified, "docker-compose.yaml"),
    (watchfiles.Change.added, "src/utils.py"),
]

print(f"\nSimulating file changes:")
for change_type, file_path in test_scenarios:
    runner.handle_change(change_type, file_path)

# Demonstrate watchfiles.run_process simulation
print(f"\nWatchfiles run_process pattern:")

def simulate_run_process():
    """Simulate watchfiles.run_process usage"""
    example_code = '''
    # Real usage would be:
    watchfiles.run_process(
        path="./src",
        target=["python", "-m", "pytest"],
        ignore_paths=["__pycache__", ".git"]
    )
    
    # Or with custom command:
    watchfiles.run_process(
        path="./frontend",
        target=["npm", "run", "build"],
        watch_filter=watchfiles.filters.PythonFilter()
    )
    '''
    
    print("Example usage:")
    print(example_code)
    
    # Simulate what would happen
    simulated_runs = [
        "python -m pytest (triggered by src/test.py change)",
        "npm run build (triggered by frontend/component.js change)",
        "python -m pytest (triggered by src/module.py change)"
    ]
    
    print("Simulated process runs:")
    for run in simulated_runs:
        print(f"  {run}")

simulate_run_process()

# Development server simulation
print(f"\nDevelopment server with auto-reload:")

class DevServer:
    """Simulate development server with auto-reload"""
    
    def __init__(self):
        self.reload_count = 0
    
    def reload(self):
        """Simulate server reload"""
        self.reload_count += 1
        print(f"  🔄 Server reloaded (reload #{self.reload_count})")
    
    async def watch_and_reload(self, watch_path):
        """Simulate watching for changes and reloading"""
        print(f"Development server watching: {watch_path}")
        
        # Simulate changes triggering reloads
        changes = ["app.py", "config.py", "templates/index.html"]
        
        for change_file in changes:
            await asyncio.sleep(0.1)
            print(f"Changed: {change_file}")
            self.reload()

dev_server = DevServer()
await dev_server.watch_and_reload(demo_dir)

# Cleanup
import shutil
shutil.rmtree(demo_dir)

print(f"\nWatchfiles enables powerful automation workflows")
print("Perfect for test runners, build systems, and development servers")

## Example 5: Production patterns and advanced file watching

In [ ]:
import watchfiles
import asyncio
import tempfile
import time
from pathlib import Path
from typing import Dict, List, Callable, Any

# Example 5: Production patterns and advanced file watching
class ProductionFileWatcher:
    """Production-ready file watching system"""
    
    def __init__(self):
        self.watchers = {}
        self.filters = {}
        self.debounce_delay = 0.1  # 100ms debounce
        self.last_events = {}
        self.stats = {
            "events_processed": 0,
            "events_debounced": 0,
            "watchers_active": 0
        }
    
    def add_watcher(self, name: str, path: str, patterns: List[str] = None, 
                   debounce: float = None):
        """Add a file watcher with configuration"""
        config = {
            "path": Path(path),
            "patterns": patterns or ["*"],
            "debounce": debounce or self.debounce_delay,
            "handlers": [],
            "active": False
        }
        
        self.watchers[name] = config
        self.last_events[name] = {}
        
        # Create appropriate filter
        if patterns:
            self.filters[name] = self._create_filter(patterns)
        else:
            self.filters[name] = watchfiles.DefaultFilter()
        
        print(f"Added watcher '{name}' for {path}")
        return config
    
    def _create_filter(self, patterns: List[str]):
        """Create custom filter based on patterns"""
        class CustomFilter(watchfiles.BaseFilter):
            def __init__(self, patterns):
                self.patterns = patterns
            
            def __call__(self, change, path):
                path_obj = Path(path)
                for pattern in self.patterns:
                    if pattern == "*":
                        return True
                    elif pattern.startswith("*."):
                        ext = pattern[1:]
                        if path_obj.suffix == ext:
                            return True
                    elif pattern.endswith("*"):
                        prefix = pattern[:-1]
                        if path_obj.name.startswith(prefix):
                            return True
                    elif pattern in str(path):
                        return True
                return False
        
        return CustomFilter(patterns)
    
    def on(self, watcher_name: str, handler: Callable):
        """Register event handler for a watcher"""
        if watcher_name in self.watchers:
            self.watchers[watcher_name]["handlers"].append(handler)
            print(f"Registered handler for '{watcher_name}'")
    
    async def _debounce_event(self, watcher_name: str, change_type, path: str):
        """Debounce rapid file changes"""
        event_key = f"{change_type.name}:{path}"
        now = time.time()
        
        # Check if this event was recently processed
        if event_key in self.last_events[watcher_name]:
            last_time = self.last_events[watcher_name][event_key]
            if now - last_time < self.watchers[watcher_name]["debounce"]:
                self.stats["events_debounced"] += 1
                return False
        
        self.last_events[watcher_name][event_key] = now
        return True
    
    async def _process_event(self, watcher_name: str, change_type, path: str):
        """Process a file system event"""
        if not await self._debounce_event(watcher_name, change_type, path):
            return
        
        self.stats["events_processed"] += 1
        config = self.watchers[watcher_name]
        
        print(f"[{watcher_name}] {change_type.name}: {Path(path).name}")
        
        # Call registered handlers
        for handler in config["handlers"]:
            try:
                if asyncio.iscoroutinefunction(handler):
                    await handler(change_type, path)
                else:
                    handler(change_type, path)
            except Exception as e:
                print(f"Handler error in {watcher_name}: {e}")
    
    async def simulate_production_watch(self, watcher_name: str, duration: float = 2.0):
        """Simulate production file watching"""
        if watcher_name not in self.watchers:
            print(f"Watcher '{watcher_name}' not found")
            return
        
        config = self.watchers[watcher_name]
        config["active"] = True
        self.stats["watchers_active"] += 1
        
        print(f"Starting production watch: {watcher_name}")
        
        # Simulate realistic file events
        events = [
            (0.1, watchfiles.Change.modified, "app.py"),
            (0.15, watchfiles.Change.modified, "app.py"),  # Rapid change (debounced)
            (0.3, watchfiles.Change.added, "new_module.py"),
            (0.5, watchfiles.Change.modified, "config.json"),
            (0.7, watchfiles.Change.deleted, "temp.log"),
            (1.0, watchfiles.Change.modified, "app.py"),
            (1.2, watchfiles.Change.added, "test_new.py"),
            (1.5, watchfiles.Change.modified, "requirements.txt"),
        ]
        
        for delay, change_type, filename in events:
            await asyncio.sleep(delay)
            if not config["active"]:
                break
            
            # Apply filter
            if self.filters[watcher_name](change_type, filename):
                await self._process_event(watcher_name, change_type, filename)
        
        config["active"] = False
        self.stats["watchers_active"] -= 1
        print(f"Stopped watching: {watcher_name}")
    
    def get_stats(self) -> Dict[str, Any]:
        """Get performance statistics"""
        return {
            **self.stats,
            "active_watchers": [name for name, config in self.watchers.items() 
                              if config.get("active", False)],
            "total_watchers": len(self.watchers)
        }

print("Production file watching patterns:")

# Create production watcher system
watcher_system = ProductionFileWatcher()

# Configure different watchers for different purposes
configs = [
    ("app_watcher", "/app/src", ["*.py", "*.yaml"], 0.2),
    ("test_watcher", "/app/tests", ["test_*.py"], 0.1),
    ("static_watcher", "/app/static", ["*.js", "*.css", "*.html"], 0.3),
    ("config_watcher", "/app/config", ["*.json", "*.yaml", "*.env"], 0.5)
]

for name, path, patterns, debounce in configs:
    watcher_system.add_watcher(name, path, patterns, debounce)

# Register production handlers
async def restart_app(change_type, path):
    """Restart application on code changes"""
    print(f"  🔄 Restarting app due to {Path(path).name}")
    await asyncio.sleep(0.01)  # Simulate restart time

def run_tests(change_type, path):
    """Run tests on test file changes"""
    print(f"  🧪 Running tests for {Path(path).name}")

async def rebuild_assets(change_type, path):
    """Rebuild static assets"""
    print(f"  🏗️  Rebuilding assets after {Path(path).name} change")
    await asyncio.sleep(0.02)  # Simulate build time

def reload_config(change_type, path):
    """Reload configuration"""
    print(f"  ⚙️  Reloading config from {Path(path).name}")

# Register handlers
watcher_system.on("app_watcher", restart_app)
watcher_system.on("test_watcher", run_tests)
watcher_system.on("static_watcher", rebuild_assets)
watcher_system.on("config_watcher", reload_config)

print(f"\nConfigured {len(configs)} production watchers")

# Run multiple watchers concurrently
print(f"\nStarting concurrent watchers:")

async def run_production_simulation():
    """Run production simulation with multiple watchers"""
    tasks = []
    
    # Start watchers with different durations
    for watcher_name in watcher_system.watchers.keys():
        task = asyncio.create_task(
            watcher_system.simulate_production_watch(watcher_name, duration=1.5)
        )
        tasks.append(task)
    
    # Wait for all watchers to complete
    await asyncio.gather(*tasks)

await run_production_simulation()

# Show performance statistics
stats = watcher_system.get_stats()
print(f"\nProduction statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

print(f"\nProduction features demonstrated:")
production_features = [
    "Multi-watcher coordination",
    "Event debouncing for performance",
    "Custom filtering patterns",
    "Async handler execution",
    "Performance monitoring",
    "Error handling and recovery"
]

for feature in production_features:
    print(f"  ✓ {feature}")

print(f"\nWatchfiles scales to enterprise file monitoring needs")
print("Ideal for CI/CD, development tools, and automation systems")